# Data Loading and Preprocessing

## 1. Split train-validation set

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

In [2]:
ROOT = Path("./CBIS-DDSM/")

In [3]:
calc_train_df = pd.read_csv(f"{ROOT}/calc_case_description_train_set.csv")
calc_test_df  = pd.read_csv(f"{ROOT}/calc_case_description_test_set.csv")

mass_train_df = pd.read_csv(f"{ROOT}/mass_case_description_train_set.csv")
mass_test_df  = pd.read_csv(f"{ROOT}/mass_case_description_test_set.csv")

training_df = pd.read_csv(f"{ROOT}/training_dataset.csv")
testing_df = pd.read_csv(f"{ROOT}/test_dataset.csv")

Convert CSV relative paths into real paths

In [4]:
def calc_train_path(csv_path):
    return Path(f"{ROOT}/CBIS-DDSM_calc_train")/Path(csv_path).parts[0]

def calc_test_path(csv_path):
    return Path(f"{ROOT}/CBIS-DDSM_calc_test")/Path(csv_path).parts[0]

def mass_train_path(csv_path):
    return Path(f"{ROOT}/CBIS-DDSM_mass_train")/Path(csv_path).parts[0]

def mass_test_path(csv_path):
    return Path(f"{ROOT}/CBIS-DDSM_mass_test")/Path(csv_path).parts[0]

calc_train_df["full_img_path_dir"] = calc_train_df["image file path"].apply(calc_train_path)
calc_train_df["mask_path_dir"] = calc_train_df["ROI mask file path"].apply(calc_train_path)

calc_test_df["full_img_path_dir"] = calc_test_df["image file path"].apply(calc_test_path)
calc_test_df["mask_path_dir"] = calc_test_df["ROI mask file path"].apply(calc_test_path)

mass_train_df["full_img_path_dir"] = mass_train_df["image file path"].apply(mass_train_path)
mass_train_df["mask_path_dir"] = mass_train_df["ROI mask file path"].apply(mass_train_path)

mass_test_df["full_img_path_dir"] = mass_test_df["image file path"].apply(mass_test_path)
mass_test_df["mask_path_dir"] = mass_test_df["ROI mask file path"].apply(mass_test_path)

Add actual file path to full mammogram images

In [5]:
# Create mapping:
# Calc-Training_P_00005_RIGHT_CC -> full image path
calc_train_case_to_path = {
    Path(case).parts[0]: str(Path(f"{ROOT}/CBIS-DDSM_calc_train")/Path(case))
    for case in training_df["full_img_path"]
}
mass_train_case_to_path = {
    Path(case).parts[0]: str(Path(f"{ROOT}/CBIS-DDSM_mass_train")/Path(case))
    for case in training_df["full_img_path"]
}
calc_test_case_to_path = {
    Path(case).parts[0]: str(Path(f"{ROOT}/CBIS-DDSM_calc_test")/Path(case))
    for case in testing_df["full_img_path"]
}
mass_test_case_to_path = {
    Path(case).parts[0]: str(Path(f"{ROOT}/CBIS-DDSM_mass_test")/Path(case))
    for case in testing_df["full_img_path"]
}

# Apply mapping
calc_train_df["full_img_path"] = (
    calc_train_df["full_img_path_dir"]
    .apply(lambda x: calc_train_case_to_path.get(Path(x).parts[2], None))
)
mass_train_df["full_img_path"] = (
    mass_train_df["full_img_path_dir"]
    .apply(lambda x: mass_train_case_to_path.get(Path(x).parts[2], None))
)
calc_test_df["full_img_path"] = (
    calc_test_df["full_img_path_dir"]
    .apply(lambda x: calc_test_case_to_path.get(Path(x).parts[2], None))
)
mass_test_df["full_img_path"] = (
    mass_test_df["full_img_path_dir"]
    .apply(lambda x: mass_test_case_to_path.get(Path(x).parts[2], None))
)

Add actual file path to ROI mask images

In [6]:
# Create mapping:
# Calc-Training_P_00005_RIGHT_CC_1 -> full image path
calc_train_case_roi_to_path = {
    Path(case).parts[0]: str(Path(f"{ROOT}/CBIS-DDSM_calc_train")/Path(case))
    for case in training_df["roi_mask_path"]
}
mass_train_case_roi_to_path = {
    Path(case).parts[0]: str(Path(f"{ROOT}/CBIS-DDSM_mass_train")/Path(case))
    for case in training_df["roi_mask_path"]
}
calc_test_case_roi_to_path = {
    Path(case).parts[0]: str(Path(f"{ROOT}/CBIS-DDSM_calc_test")/Path(case))
    for case in testing_df["roi_mask_path"]
}
mass_test_case_roi_to_path = {
    Path(case).parts[0]: str(Path(f"{ROOT}/CBIS-DDSM_mass_test")/Path(case))
    for case in testing_df["roi_mask_path"]
}

# Apply mapping
calc_train_df["roi_mask_path"] = (
    calc_train_df["mask_path_dir"]
    .apply(lambda x: calc_train_case_roi_to_path.get(Path(x).parts[2], None))
)
mass_train_df["roi_mask_path"] = (
    mass_train_df["mask_path_dir"]
    .apply(lambda x: mass_train_case_roi_to_path.get(Path(x).parts[2], None))
)
calc_test_df["roi_mask_path"] = (
    calc_test_df["mask_path_dir"]
    .apply(lambda x: calc_test_case_roi_to_path.get(Path(x).parts[2], None))
)
mass_test_df["roi_mask_path"] = (
    mass_test_df["mask_path_dir"]
    .apply(lambda x: mass_test_case_roi_to_path.get(Path(x).parts[2], None))
)

Remove all rows containing any null value in any column from the dataframes

In [7]:
calc_train_df = calc_train_df.dropna().reset_index(drop=True)
mass_train_df = mass_train_df.dropna().reset_index(drop=True)
calc_test_df = calc_test_df.dropna().reset_index(drop=True)
mass_test_df = mass_test_df.dropna().reset_index(drop=True)

Add actual file path to cropped images

In [8]:
def get_cropped_image_path(roi_mask_path):
    roi_path = Path(roi_mask_path)
    
    # Folder containing both files
    folder = roi_path.parent
    
    # Get all DICOM files in folder
    dcm_files = sorted(folder.glob("*.dcm"))
        
    if len(dcm_files) != 2:
        print(f"Warning: {roi_path.parent} contains {len(dcm_files)} DICOM files")
        return None

    # Find the file that is NOT the ROI mask
    cropped_file = next(f for f in dcm_files if f.name != roi_path.name)

    return str(cropped_file)

calc_train_df["crop_img_path"] = (
    calc_train_df["roi_mask_path"]
    .apply(get_cropped_image_path)
)
mass_train_df["crop_img_path"] = (
    mass_train_df["roi_mask_path"]
    .apply(get_cropped_image_path)
)
calc_test_df["crop_img_path"] = (
    calc_test_df["roi_mask_path"]
    .apply(get_cropped_image_path)
)
mass_test_df["crop_img_path"] = (
    mass_test_df["roi_mask_path"]
    .apply(get_cropped_image_path)
)

Remove all rows containing null value in crop_img_path column

In [9]:
calc_train_df = calc_train_df.dropna(subset=["crop_img_path"]).reset_index(drop=True)
mass_train_df = mass_train_df.dropna(subset=["crop_img_path"]).reset_index(drop=True)
calc_test_df = calc_test_df.dropna(subset=["crop_img_path"]).reset_index(drop=True)
mass_test_df = mass_test_df.dropna(subset=["crop_img_path"]).reset_index(drop=True)

Save updated metadata files

In [10]:
calc_train_df.to_csv(f"{ROOT}/new_calc_train_metadata.csv", index=False)
mass_train_df.to_csv(f"{ROOT}/new_mass_train_metadata.csv", index=False)
calc_test_df.to_csv(f"{ROOT}/new_calc_test_metadata.csv", index=False)
mass_test_df.to_csv(f"{ROOT}/new_mass_test_metadata.csv", index=False)

Combine respective calc and mass metadata

In [11]:
train_df = pd.concat([calc_train_df, mass_train_df], ignore_index=True)
test_df = pd.concat([calc_test_df, mass_test_df], ignore_index=True)

In [12]:
print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (2329, 21)
Test: (608, 22)


Patient-level train-validation split

In [13]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2, # 20% of train set
    random_state=42
)

train_idx, val_idx = next(
    gss.split(
        train_df,
        groups=train_df["patient_id"]
    )
)

final_train_df = train_df.iloc[train_idx].reset_index(drop=True)
val_df = train_df.iloc[val_idx].reset_index(drop=True)

In [14]:
train_patients = set(final_train_df["patient_id"])
val_patients = set(val_df["patient_id"])
test_patients = set(test_df["patient_id"])

print("Train-Val overlap:", len(train_patients & val_patients))
print("Train-Test overlap:", len(train_patients & test_patients))
print("Val-Test overlap:", len(val_patients & test_patients))

Train-Val overlap: 0
Train-Test overlap: 21
Val-Test overlap: 5


Remove overlapping patients from training/validation metadata

In [15]:
# Identify patients that appear in official test set
test_patients = set(test_df["patient_id"])

# Remove any train/val rows whose patient_id appears in test
final_train_df_clean = final_train_df[
    ~final_train_df["patient_id"].isin(test_patients)
].reset_index(drop=True)

val_df_clean = val_df[
    ~val_df["patient_id"].isin(test_patients)
].reset_index(drop=True)

In [16]:
train_patients = set(final_train_df_clean["patient_id"])
val_patients = set(val_df_clean["patient_id"])
test_patients = set(test_df["patient_id"])

print("Train-Val overlap:", len(train_patients & val_patients))
print("Train-Test overlap:", len(train_patients & test_patients))
print("Val-Test overlap:", len(val_patients & test_patients))

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


In [17]:
print("Final train:", final_train_df_clean.shape)
print("Validation:", val_df_clean.shape)
print("Test:", test_df.shape)

Final train: (1776, 21)
Validation: (498, 21)
Test: (608, 22)


Save metadata files

In [18]:
print("Train class distribution:")
print(final_train_df_clean["abnormality type"].value_counts())

print("\nValidation class distribution:")
print(val_df_clean["abnormality type"].value_counts())

print("\nTest class distribution:")
print(test_df["abnormality type"].value_counts())

Train class distribution:
abnormality type
mass             942
calcification    834
Name: count, dtype: int64

Validation class distribution:
abnormality type
calcification    279
mass             219
Name: count, dtype: int64

Test class distribution:
abnormality type
mass             349
calcification    259
Name: count, dtype: int64


In [19]:
final_train_df_clean.to_csv(f"{ROOT}/combined_train_metadata.csv", index=False)
val_df_clean.to_csv(f"{ROOT}/combined_val_metadata.csv", index=False)
test_df.to_csv(f"{ROOT}/combined_test_metadata.csv", index=False)

## 2. Read DICOM files and convert ROI masks to full-image bounding box coordinates

In [20]:
import pydicom
import numpy as np
import cv2

Function to read DICOM image

In [21]:
def load_dicom_image(dicom_path):
    ds = pydicom.dcmread(dicom_path)
    img = ds.pixel_array.astype(np.float32)

    if ds.PhotometricInterpretation == "MONOCHROME1":
        img = img.max() - img

    return img

Function to get bounding box from ROI mask

In [22]:
def mask_to_bbox(mask):
    ys, xs = np.where(mask > 0)

    if len(xs) == 0 or len(ys) == 0:
        return None

    x_min = xs.min()
    x_max = xs.max() + 1
    y_min = ys.min()
    y_max = ys.max() + 1

    return [x_min, y_min, x_max, y_max]

## 3. Percentile intensity clipping

Function to perform percentile clipping

In [23]:
def percentile_clip(img, lower=1, upper=99):
    p_low = np.percentile(img, lower)
    p_high = np.percentile(img, upper)

    img = np.clip(img, p_low, p_high)
    
    return img

## 4. Normalise pixel values

Function to normalise pixel values to 0-1

In [24]:
def normalize_0_to_1(img):
    img_min = img.min()
    img_max = img.max()

    img = (img - img_min) / (img_max - img_min + 1e-6)

    return img.astype(np.float32)

## 5. Replicate grayscale image into three channels

Function to convert grayscale to 3 channels

In [25]:
def to_three_channels(img):
    return np.stack([img, img, img], axis=-1)

## 6. Letterbox resize

Function to perform letterbox resize on full image and adjust all bounding boxes

In [26]:
def letterbox(img, bboxes, new_size=1024):
    h, w = img.shape[:2]

    scale = min(new_size / w, new_size / h)

    resized_w = int(w * scale)
    resized_h = int(h * scale)

    resized_img = cv2.resize(img, (resized_w, resized_h))

    canvas = np.zeros((new_size, new_size, 3), dtype=np.float32)

    pad_x = (new_size - resized_w) // 2
    pad_y = (new_size - resized_h) // 2

    canvas[pad_y:pad_y + resized_h, pad_x:pad_x + resized_w] = resized_img

    # Update all bounding boxes after resizing
    new_bboxes = []
    for bbox in bboxes:
        x_min, y_min, x_max, y_max = bbox[:4]
        new_bboxes.append([
            x_min * scale + pad_x,
            y_min * scale + pad_y,
            x_max * scale + pad_x,
            y_max * scale + pad_y
        ])

    return canvas, new_bboxes

## 7. Convert to bounding box coordinates YOLO format

Create class mapping to get class ID

In [28]:
CLASS_MAP_2c = {
    ("calcification", "BENIGN"): 0,
    ("calcification", "BENIGN_WITHOUT_CALLBACK"): 0,
    ("calcification", "MALIGNANT"): 1,
    ("mass", "BENIGN"): 0,
    ("mass", "BENIGN_WITHOUT_CALLBACK"): 0,
    ("mass", "MALIGNANT"): 1,
}

Function to get class ID

In [29]:
def get_class_id(row, class_map):
    abnormality = row["abnormality type"].strip().lower()
    pathology = row["pathology"].strip().upper()

    key = (abnormality, pathology)

    if key not in class_map:
        raise ValueError(f"Unknown class combination: {key}")

    return class_map[key]

Function to convert bounding box to YOLO format

In [30]:
def bbox_to_yolo(class_id, bbox, img_size=1024):
    x_min, y_min, x_max, y_max = bbox

    x_center = ((x_min + x_max) / 2) / img_size
    y_center = ((y_min + y_max) / 2) / img_size
    width = (x_max - x_min) / img_size
    height = (y_max - y_min) / img_size

    return [class_id, x_center, y_center, width, height]

Function to save processed image and YOLO label

In [31]:
def save_processed_sample(img, yolo_labels, image_output_path, label_output_path):
    # Convert 0–1 image to 0–255 PNG
    img_uint8 = (img * 255).astype(np.uint8)

    cv2.imwrite(str(image_output_path), img_uint8)

    with open(label_output_path, "w") as f:
        for yolo_label in yolo_labels:
            f.write(
                f"{int(yolo_label[0])} "
                f"{yolo_label[1]:.6f} "
                f"{yolo_label[2]:.6f} "
                f"{yolo_label[3]:.6f} "
                f"{yolo_label[4]:.6f}\n"
            )

## 8. Full preprocessing

Function to process one full mammogram with multiple lesions

In [32]:
def preprocess_full_mammogram_case(
    case_rows,
    class_map,
    image_output_path,
    label_output_path,
    img_size=1024,
):
    # Load full image
    full_image_path = case_rows.iloc[0]["full_img_path"]
    full_img = load_dicom_image(full_image_path)

    bboxes = []
    for _, row in case_rows.iterrows():

        # Load ROI mask
        roi_mask = load_dicom_image(row["roi_mask_path"])

        if roi_mask.shape != full_img.shape:
            raise ValueError(
                f"Mask shape {roi_mask.shape} does not match full image shape {full_img.shape}"
            )

        # Get full-image bounding box
        bbox = mask_to_bbox(roi_mask)

        if bbox is None:
            print("No ROI found:", full_image_path)
            continue

        # Get class ID for each bounding box
        class_id = get_class_id(row, class_map)

        x_min, y_min, x_max, y_max = bbox
        bboxes.append([
            x_min,
            y_min,
            x_max,
            y_max,
            class_id
        ])

    if len(bboxes) == 0:
        print("No valid boxes:", full_image_path)
        return

    # Percentile intensity clipping
    full_img = percentile_clip(full_img)

    # Normalise pixel values
    full_img = normalize_0_to_1(full_img)

    # Replicate grayscale image into three channels
    full_img = to_three_channels(full_img)

    # Letterbox resize
    full_img, bboxes = letterbox(
        full_img,
        bboxes,
        new_size=img_size
    )

    # Convert bounding box coordinates to YOLO format
    yolo_labels = [
        bbox_to_yolo(class_id, bbox, img_size=img_size)
        for bbox in bboxes
    ]

    for label in yolo_labels:
        _, x, y, w, h = label
        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 <= w <= 1 and 0 <= h <= 1):
            raise ValueError(f"Invalid YOLO label: {label}")

    # Save processed image and label
    save_processed_sample(
        full_img,
        yolo_labels,
        image_output_path,
        label_output_path
    )

Function to process each metadata CSV file

In [33]:
def process_metadata_csv(
    metadata_csv,
    class_map,
    image_output_dir,
    label_output_dir,
    img_size=1024
):
    df = pd.read_csv(metadata_csv)

    image_output_dir = Path(image_output_dir)
    label_output_dir = Path(label_output_dir)

    image_output_dir.mkdir(parents=True, exist_ok=True)
    label_output_dir.mkdir(parents=True, exist_ok=True)

    # Group rows belonging to the same full mammogram
    grouped = df.groupby("full_img_path_dir")

    total = len(grouped)

    for idx, (image_path, case_rows) in enumerate(grouped, start=1):
        case_name = Path(image_path).name
        
        try:
            image_output_path = image_output_dir / f"{case_name}.png"
            label_output_path = label_output_dir / f"{case_name}.txt"

            preprocess_full_mammogram_case(
                case_rows=case_rows,
                class_map=class_map,
                image_output_path=image_output_path,
                label_output_path=label_output_path,
                img_size=img_size
            )

            print(f"[{idx}/{total}] Processed {case_name}")
        except Exception as e:
            print(f"[{idx}/{total}] Failed: {image_path}")
            print(e)

Execute preprocessing

In [35]:
OUT_DIR = Path("./YOLO_labelled_2c")
class_map = CLASS_MAP_2c

In [35]:
splits = [
    (f"{ROOT}/combined_train_metadata.csv", "train"),
    (f"{ROOT}/combined_val_metadata.csv", "val"),
    (f"{ROOT}/combined_test_metadata.csv", "test")
]

for csv_file, split in splits:
    process_metadata_csv(
        metadata_csv=csv_file,
        class_map=class_map,
        image_output_dir=f"{OUT_DIR}/{split}/images/",
        label_output_dir=f"{OUT_DIR}/{split}/labels/",
        img_size=1024
    )

[1/1652] Processed Calc-Training_P_00005_RIGHT_CC
[2/1652] Processed Calc-Training_P_00005_RIGHT_MLO
[3/1652] Processed Calc-Training_P_00010_LEFT_CC
[4/1652] Processed Calc-Training_P_00010_LEFT_MLO
[5/1652] Processed Calc-Training_P_00011_LEFT_CC
[6/1652] Processed Calc-Training_P_00011_LEFT_MLO
[7/1652] Processed Calc-Training_P_00012_LEFT_CC
[8/1652] Processed Calc-Training_P_00012_LEFT_MLO
[9/1652] Processed Calc-Training_P_00013_RIGHT_MLO
[10/1652] Processed Calc-Training_P_00014_LEFT_CC
[11/1652] Processed Calc-Training_P_00014_LEFT_MLO
[12/1652] Processed Calc-Training_P_00019_RIGHT_CC
[13/1652] Processed Calc-Training_P_00019_RIGHT_MLO
[14/1652] Processed Calc-Training_P_00020_LEFT_CC
[15/1652] Processed Calc-Training_P_00020_LEFT_MLO
[16/1652] Processed Calc-Training_P_00022_LEFT_CC
[17/1652] Processed Calc-Training_P_00022_LEFT_MLO
[18/1652] Processed Calc-Training_P_00024_LEFT_CC
[19/1652] Processed Calc-Training_P_00024_LEFT_MLO
[20/1652] Processed Calc-Training_P_00028_LE